In [ ]:
# ===== Base =====
!apt -y install -qq aria2
%cd /content

from pathlib import Path
# ===== ComfyUI =====
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!git pull
!pip install -q -r requirements.txt
!pip install color-matcher

# -----  sageattentionのインストール 処理開始
# 0) ビルド系を用意
!pip install -U pip setuptools wheel ninja packaging cmake

# 1) CUDA ツールキットの場所を明示（Colabは通常 /usr/local/cuda）
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"

# 2) SageAttention 2.2.0 をビルド分離OFFで入れる（verbose付き）
!pip install "sageattention==2.2.0" --no-build-isolation -v

#  -----  sageattentionのインストール 処理終了

# ===== Custom nodes =====
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git || true
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git || true
!git clone https://github.com/cubiq/ComfyUI_essentials || true
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git || true
!git clone https://github.com/kijai/ComfyUI-KJNodes || true
!git clone https://github.com/T8mars/comfyui-purgevram.git || true
!git clone https://github.com/rgthree/rgthree-comfy || true
%cd rgthree-comfy; !pip install -q -r requirements.txt; %cd ..

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git || true
%cd ComfyUI-WanVideoWrapper
!pip install -q -r requirements.txt
%cd ..

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-segment-anything-2.git || true
%cd ComfyUI-segment-anything-2
!pip install -q -r requirements.txt
%cd ..

!git clone https://github.com/city96/ComfyUI-GGUF.git || true
%cd ComfyUI-GGUF
!pip install -q -r requirements.txt
%cd ..

# controlnet-aux 依存
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/requirements.txt

# ===== Folders =====
%cd /content/ComfyUI
for p in [
    "models/diffusion_models", "models/loras",
    "models/text_encoders", "models/vae",
    "models/clip_vision",
    "user/default/workflows", "input"
]:
    Path(p).mkdir(parents=True, exist_ok=True)

# ===== Download helper =====
import os, sys
def dl(url, out_path):
    out_dir = os.path.dirname(out_path)
    os.makedirs(out_dir, exist_ok=True)
    cmd = f'aria2c --console-log-level=error -c -x16 -s16 -k1M "{url}" -d "{out_dir}" -o "{os.path.basename(out_path)}"'
    code = os.system(cmd)
    if code != 0:
        print("DOWNLOAD FAILED:", url, "->", out_path)
    else:
        print("DOWNLOADED:", out_path)

# ====== 指定の Download Link を反映 ======
DL = [
    # --- ComfyUI/models/diffusion_models ---
    # Wan22Animate
    # ("https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/Wan22Animate/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors",
    #  "models/diffusion_models/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-I2V-14B-480P_fp8_e4m3fn.safetensors",
     "models/diffusion_models/Wan2_1-I2V-14B-480P_fp8_e4m3fn.safetensors"),

    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-I2V-14B-480P_fp8_e5m2.safetensors",
     "models/diffusion_models/Wan2_1-I2V-14B-480P_fp8_e5m2.safetensors"),


    # ===== Stable-Video-Infinity LoRAs =====
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-dance_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-dance_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-film-opt-10212025_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-film-opt-10212025_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-film-transitions_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-film-transitions_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-film_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-film_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-shot_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-shot_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-talk_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-talk_lora_rank_128_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/svi-tom_lora_rank_128_fp16.safetensors",
     "models/loras/Stable-Video-Infinity/svi-tom_lora_rank_128_fp16.safetensors"),

    # Improving Quality (T2V)
    ("https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/T2V/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
     "models/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/T2V/Wan2_2-T2V-A14B-LOW_fp8_e5m2_scaled_KJ.safetensors",
     "models/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e5m2_scaled_KJ.safetensors"),

    # Flux Krea
    ("https://huggingface.co/Comfy-Org/FLUX.1-Krea-dev_ComfyUI/resolve/main/split_files/diffusion_models/flux1-krea-dev_fp8_scaled.safetensors",
     "models/diffusion_models/flux1-krea-dev_fp8_scaled.safetensors"),

    # --- ComfyUI/models/text_encoders ---
    ("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
     "models/text_encoders/clip_l.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5-xxl-enc-bf16.safetensors",
     "models/text_encoders/umt5-xxl-enc-bf16.safetensors"),
    ("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors",
     "models/text_encoders/t5xxl_fp16.safetensors"),

    # --- ComfyUI/models/clip_vision ---
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
     "models/clip_vision/clip_vision_h.safetensors"),

    # --- ComfyUI/models/vae ---
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1_VAE_bf16.safetensors",
     "models/vae/Wan2_1_VAE_bf16.safetensors"),
    ("https://huggingface.co/Comfy-Org/Lumina_Image_2.0_Repackaged/resolve/main/split_files/vae/ae.safetensors",
     "models/vae/ae.safetensors"),

    # --- ComfyUI/models/loras ---
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank128_bf16.safetensors",
     "models/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank128_bf16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/WanAnimate_relight_lora_fp16.safetensors",
     "models/loras/WanAnimate_relight_lora_fp16.safetensors"),
]

for url, path in DL:
    dl(url, f"/content/ComfyUI/{path}")

# --------------------- 任意：デフォルトのworkflow/サンプル入力（必要なら） ---------------------
# 下記はユーザー様の既存スクリプトを踏襲。不要なら削除可。
prefix = "0105"
uuid = "dc2c4ad1-9fb4-4957-ab8c-def83ab515ba"
root_path = f"https://archive.creativaier.com/comfyui_materials/{prefix}_{uuid}"

import requests, json
try:
    url = f"{root_path}/workflow.json"
    data = requests.get(url).json()
    Path("/content/ComfyUI/user/default/workflows").mkdir(parents=True, exist_ok=True)
    with open("/content/ComfyUI/user/default/workflows/default.json", "w") as f:
        f.write(json.dumps(data, indent=2))
    # 入力素材
    !aria2c --console-log-level=error -c -x16 -s16 -k1M "{root_path}/photo.png" -d /content/ComfyUI/input -o photo.png
    print("📦 Default workflow & sample media pulled.")
except Exception as e:
    print("Workflow/material fetch skipped or failed:", e)

# ===== Cloudflared =====
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb || true

import subprocess, threading, time, socket
def wait_and_tunnel(port=8188):
    while True:
        time.sleep(0.5)
        try:
            s = socket.create_connection(("127.0.0.1", port), timeout=0.5)
            s.close(); break
        except Exception:
            pass
    print("\nComfyUI is up. Launching Cloudflared tunnel...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in p.stderr:
        if "trycloudflare.com " in line:
            print("External URL:", line[line.find("http"):].strip())

threading.Thread(target=wait_and_tunnel, kwargs={"port": 8188}, daemon=True).start()

# ===== Run ComfyUI =====
!python main.py --dont-print-server --listen 0.0.0.0 --port 8188
